In [ ]:
# Based on the following works:

# Muñoz-Sierra, R., Ayensa-Jiménez, J., & Doblaré, M. (2025).
# On the application of physically-guided neural networks with internal variables to continuum problems.
# Mechanics of Materials, 205, 105317.
# https://doi.org/10.1016/j.mechmat.2025.105317

# Ayensa-Jiménez, J., Orera-Echeverría, J., & Doblaré, M. (2025).
# Predicting and explaining nonlinear material response using deep physically guided neural networks with internal variables.
# Mathematics and Mechanics of Solids, 30(2), 573–598.
# https://doi.org/10.1177/10812865241257850

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()))))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'model')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '../../src')))

In [ ]:
import torch
import numpy as np 

# TensOps library
import tensopsciml as to
from tensopsciml.tensors import TensOps

# Functions of PGNNIV example
from architecture.PGNNIV_architecture import PGNNIV
from utils.folders import create_folder
from utils.load_data import load_data
from utils.split_data import get_train_val_indices, train_val_split_torch_tensops
from trainers.train import train_loop

In [ ]:
SEED = 42

np.random.seed(SEED)  
torch.manual_seed(SEED)  
GEN = torch.Generator().manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

if DEVICE.type == 'cuda':
    torch.cuda.manual_seed(SEED)

Load data

In [ ]:
data_name = 'Ogden_N1000_S0_train'
model_name = 'PGNNIV_baseline'

# Paths definition
ROOT_PATH = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '../pgnniv'))
DATA_FOLDER_PATH = os.path.join(ROOT_PATH, 'datasets', 'train_data', 'data')
RESULTS_FOLDER_PATH = os.path.join(ROOT_PATH, 'results')

DATA_PATH = os.path.join(DATA_FOLDER_PATH, data_name) + '.pkl'
MODEL_RESULTS_PATH = os.path.join(RESULTS_FOLDER_PATH, model_name, data_name)

# Folders creation (if necessary)   
create_folder(RESULTS_FOLDER_PATH)
create_folder(MODEL_RESULTS_PATH)

In [ ]:
dataset = load_data(DATA_PATH)

In [ ]:
N_DATA = dataset['N_DATA']
TRAIN_SPLIT = 0.7
VAL_SPLIT = 1 - TRAIN_SPLIT

dx = dataset['x_step_size']
dy = dataset['y_step_size']
D = to.Derivative(dx, dy, 0, indexing='xy').two_dimensions().to(DEVICE)

In [ ]:
U_dataset = dataset['U']
F_dataset = dataset['F']
E_dataset = dataset['E']
P_dataset = dataset['P']
Tx_dataset, Ty_dataset = dataset['Tx'], dataset['Ty']
lambda1, lambda2, lambda3 = dataset['lambda1'], dataset['lambda2'], dataset['lambda3']

In [ ]:
# Displacement field U(x, y)
U = torch.zeros(U_dataset.shape[0], 2, U_dataset.shape[1], U_dataset.shape[2], dtype=torch.float32)
U[:, 0] = torch.tensor(U_dataset[..., 0], dtype=torch.float32)
U[:, 1] = torch.tensor(U_dataset[..., 1], dtype=torch.float32)
U = TensOps(U.to(DEVICE), domain_dim=2, codomain_dim=[2], order=[1, 0], contravariant_indices=[0], covariant_indices=None)

# Gradient field FF(x, y)
F = torch.zeros(F_dataset.shape[0], 2, 2, F_dataset.shape[1]-1, F_dataset.shape[2]-1)
F[:, 0, 0] = to.m2(to.m1(TensOps(torch.tensor(F_dataset[..., 0], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
F[:, 1, 1] = to.m2(to.m1(TensOps(torch.tensor(F_dataset[..., 1], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
F = TensOps(F.to(DEVICE), domain_dim=2, codomain_dim=[2, 2], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])

# Green-Lagrange tensor EE(x, y)
E = torch.zeros(E_dataset.shape[0], 2, 2, E_dataset.shape[1]-1, E_dataset.shape[2]-1)
E[:, 0, 0] = to.m2(to.m1(TensOps(torch.tensor(E_dataset[..., 0], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
E[:, 1, 1] = to.m2(to.m1(TensOps(torch.tensor(E_dataset[..., 1], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
E = TensOps(E.to(DEVICE), domain_dim=2, codomain_dim=[2, 2], order=[1, 1], contravariant_indices=[0], covariant_indices=[1])

# First Piola-Kirchhoff stress P(x, y). P^{ij}, order [2, 0].
P = torch.zeros(P_dataset.shape[0], 2, 2, P_dataset.shape[1]-1, P_dataset.shape[2]-1)
P[:, 0, 0] = to.m2(to.m1(TensOps(torch.tensor(P_dataset[..., 0], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
P[:, 1, 1] = to.m2(to.m1(TensOps(torch.tensor(P_dataset[..., 1], dtype=torch.float32).unsqueeze(1), domain_dim=2, codomain_dim=[1], order=[0, 0], contravariant_indices=None, covariant_indices=None))).values[:, 0]
P = TensOps(P.to(DEVICE), domain_dim=2, codomain_dim=[2, 2], order=[2, 0], contravariant_indices=[0, 1], covariant_indices=None)

# Boundary tractions T^i = P^{ij} N_j (full traction vectors), already
# cell-averaged and reshaped to (N, 2, Ny-1, 1) / (N, 2, 1, Nx-1) by the data
# generator.
Tx = torch.tensor(Tx_dataset, dtype=torch.float32)
Ty = torch.tensor(Ty_dataset, dtype=torch.float32)

In [ ]:
train_idx, val_idx = get_train_val_indices(N_DATA, TRAIN_SPLIT, GEN)

U_train, U_val = train_val_split_torch_tensops(U, train_idx, val_idx)
F_train, F_val = train_val_split_torch_tensops(F, train_idx, val_idx)
E_train, E_val = train_val_split_torch_tensops(E, train_idx, val_idx)
P_train, P_val = train_val_split_torch_tensops(P, train_idx, val_idx)
Tx_train, Tx_val = train_val_split_torch_tensops(Tx, train_idx, val_idx)
Ty_train, Ty_val = train_val_split_torch_tensops(Ty, train_idx, val_idx)

lambda1_train, lambda1_val = train_val_split_torch_tensops(torch.tensor(lambda1), train_idx, val_idx)
lambda2_train, lambda2_val = train_val_split_torch_tensops(torch.tensor(lambda2), train_idx, val_idx)
lambda3_train, lambda3_val = train_val_split_torch_tensops(torch.tensor(lambda3), train_idx, val_idx)

In [ ]:
# Model parameters
latent_space_predictive = 10
n_filters_explanatory = 10

# Predictive network architecture
input_shape_1 = Tx[0].shape
input_shape_2 = Ty[0].shape
predictive_layers = [50, 20, latent_space_predictive, 20, 50]
predictive_output = U.values[0].shape

# Explanatory network architecture
explanatory_input = E.values[0].shape
explanatory_layers = [50]
explanatory_output = P.values[0].shape

# Metric of the problem: Euclidean (Cartesian) g_{ij} = delta_{ij} on the
# interior grid where GradU/E/P live. Built once here as a batch-1 MetrOps and
# passed into the model, which broadcasts it to each step's batch size.
grid = E.values.shape[-2:]
metric_ = torch.eye(2, dtype=torch.float32).reshape(1, 2, 2, 1, 1).expand(1, 2, 2, *grid).clone()
metric = to.MetrOps(metric_.to(DEVICE), domain_dim=2, codomain_dim=[2, 2])

In [ ]:
# Characteristic stress/traction scale (~1e5). Used both to non-dimensionalize
# the encoder inputs (so its sigmoids do not saturate) and to lift the
# explanatory network's O(1) output up to the physical stress magnitude, so R3
# (whose target tractions are ~1e5) becomes reachable at Adam's step size.
# Use the traction-vector NORM (over the 2 components) so the identically-zero
# shear component does not drag the mean down.
STRESS_SCALE = torch.cat([Tx_train.norm(dim=1).flatten(), Ty_train.norm(dim=1).flatten()]).mean().item()
print(f"Stress/traction scale: {STRESS_SCALE:.3e}")

pgnniv_model = PGNNIV(input_shape_1, input_shape_2, predictive_layers, predictive_output,
                      explanatory_input, explanatory_layers, explanatory_output,
                      n_filters_explanatory, D, metric,
                      input_scale=STRESS_SCALE, output_scale=STRESS_SCALE).to(DEVICE)

optimizer = torch.optim.Adam(pgnniv_model.parameters(), lr=1e-3)

In [ ]:
# Training parameters
start_epoch = 0
n_epochs = 10000

batch_size = 64
n_checkpoints = 10

train_loop(pgnniv_model, optimizer, Tx_train, Ty_train, U_train, Tx_val, Ty_val, U_val,
           D, n_checkpoints, start_epoch=start_epoch, n_epochs=n_epochs, batch_size=batch_size, 
           model_results_path=MODEL_RESULTS_PATH, device=DEVICE, new_lr=1e-5)